# Nassau Candy Distributor — Factory Reallocation & Shipping Optimization
## Steps 1–6: Data Inspection, Cleaning, Feature Engineering, EDA, Preparation and Model Training

**Project goal:** build an internship-level decision-support system that predicts order-to-ship lead time and later uses those predictions for factory what-if analysis.

This notebook deliberately stops after training the three required models:
1. Linear Regression
2. Random Forest Regressor
3. Gradient Boosting Regressor

> **Important data limitation:** the supplied dataset does not contain a historical factory-assignment column, transportation cost, or factory capacity. Therefore, the ML model in this notebook learns lead-time patterns from the fields that actually exist in the dataset. Factory-specific scenario logic will be designed separately and will not pretend that historical factory effects were learned when they were not observed.

## 1. Load the dataset

We first load the original CSV and inspect its dimensions and column names. We do not manually alter the source file.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

DATA_PATH = Path("../data/Nassau_Candy.csv")

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

## 2. Inspect the dataset

The first checks answer four basic questions:

- What data types are present?
- Are there missing values?
- Are there exact duplicate rows?
- Are the categorical fields reasonable?

IDs are kept in the raw data for traceability, but we will **not use Row ID, Order ID or Customer ID as predictive features**, because they identify records/customers rather than describe the shipping situation.

In [ ]:
print(df.info())

print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False))

print("\nExact duplicate rows:", df.duplicated().sum())

print("\nUnique values in important categorical fields:")
for col in ["Ship Mode", "Country/Region", "Division", "Region", "Product Name"]:
    print(f"\n{col}: {df[col].nunique()}")
    print(df[col].value_counts().head(20))

## 3. Data cleaning and validation

### 3.1 Parse dates

`Order Date` and `Ship Date` are converted from text into real datetime values.

### 3.2 Standardize the product-name typo

The dataset contains both:

- `Wonka Bar - Scrumdiddlyumptious`
- `Wonka Bar -Scrumdiddlyumptious`

These refer to the same product, so they are standardized.

### 3.3 Create the target

`Lead Time = Ship Date - Order Date`, measured in days.

We **do not remove the unusual 900–1,600 day values**. They are present systematically across the dataset rather than appearing as a few random mistakes. Removing them would change the business problem and could make the model look artificially better.

In [ ]:
df = df.copy()

# Convert dates
df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst=True, errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], dayfirst=True, errors="coerce")

# Standardize known product-name inconsistency
df["Product Name"] = df["Product Name"].replace({
    "Wonka Bar -Scrumdiddlyumptious": "Wonka Bar - Scrumdiddlyumptious"
})

# Create target
df["Lead Time"] = (df["Ship Date"] - df["Order Date"]).dt.days

print("Invalid Order Dates:", df["Order Date"].isna().sum())
print("Invalid Ship Dates:", df["Ship Date"].isna().sum())
print("Negative Lead Times:", (df["Lead Time"] < 0).sum())
print("Zero Lead Times:", (df["Lead Time"] == 0).sum())

print("\nLead-time summary:")
print(df["Lead Time"].describe())

### 3.4 Validate financial fields

The supplied data contains `Sales`, `Cost`, and `Gross Profit`.

We check whether:

`Gross Profit ≈ Sales - Cost`

This is useful because the later recommendation engine will use historical gross margin as a profitability indicator.

In [ ]:
profit_difference = (df["Sales"] - df["Cost"] - df["Gross Profit"]).abs()

print("Maximum absolute difference:", profit_difference.max())

df["Profit Margin"] = np.where(
    df["Sales"] != 0,
    df["Gross Profit"] / df["Sales"],
    np.nan
)

print("\nProfit margin summary:")
print(df["Profit Margin"].describe())

### 3.5 Duplicates and identifier checks

Repeated `Order ID`s are not automatically errors because an order can contain multiple product lines.

`Product ID` repeats by design because many orders contain the same product.

We therefore only remove **exact duplicate rows**, and there are none in the supplied data.

In [ ]:
print("Exact duplicate rows:", df.duplicated().sum())

for col in ["Row ID", "Order ID", "Customer ID", "Product ID"]:
    print(f"{col}: {df[col].nunique()} unique values out of {len(df)} rows")

print("\nCleaned shape:", df.shape)

## 4. Missing values, outliers and lead-time investigation

A key finding is that lead time is **multimodal** rather than normally distributed. It is concentrated around three broad bands:

- roughly 904–915 days
- roughly 1,269–1,277 days
- roughly 1,634–1,642 days

These are not isolated statistical outliers. They occur repeatedly and are also associated with different order/ship-year combinations. We therefore retain them and document the unusual structure as a dataset characteristic.

For Sales, Units, Gross Profit and Cost, IQR flags are reported for inspection, but observations are not automatically deleted.

In [ ]:
print("Lead-time quantiles:")
print(df["Lead Time"].quantile([0, .01, .05, .25, .50, .75, .95, .99, 1]))

print("\nIQR outlier counts (inspection only):")
for col in ["Sales", "Units", "Gross Profit", "Cost", "Lead Time"]:
    q1 = df[col].quantile(.25)
    q3 = df[col].quantile(.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: {count}")

print("\nLead-time frequency:")
print(df["Lead Time"].value_counts().sort_index().head(30))

## 5. Date processing and feature engineering

The model should use information that would reasonably be available when an order is being evaluated.

We derive:

- Order Year
- Order Month
- Order Day of Month
- Order Day of Week
- Order Quarter

We intentionally exclude `Ship Date`, because it occurs **after** the prediction point and would leak the answer.

We also exclude Row ID, Order ID and Customer ID from the ML features.

`Sales`, `Units`, `Gross Profit` and `Cost` are retained because they are transaction-level attributes available in the supplied order data.

In [ ]:
df["Order Year"] = df["Order Date"].dt.year
df["Order Month"] = df["Order Date"].dt.month
df["Order DayOfMonth"] = df["Order Date"].dt.day
df["Order DayOfWeek"] = df["Order Date"].dt.dayofweek
df["Order Quarter"] = df["Order Date"].dt.quarter

feature_columns = [
    "Ship Mode",
    "Country/Region",
    "City",
    "State/Province",
    "Postal Code",
    "Division",
    "Region",
    "Product ID",
    "Product Name",
    "Sales",
    "Units",
    "Gross Profit",
    "Cost",
    "Order Year",
    "Order Month",
    "Order DayOfMonth",
    "Order DayOfWeek",
    "Order Quarter"
]

target = "Lead Time"

print("Number of model features:", len(feature_columns))
print("\nFeatures:")
print(feature_columns)

## 6. Exploratory Data Analysis (EDA)

The first plots focus on the variables most relevant to the shipping problem.

The EDA is intentionally practical rather than decorative: we want to understand the target distribution, operational categories and profitability before modeling.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

sns.histplot(df["Lead Time"], bins=40, kde=False, ax=axes[0, 0])
axes[0, 0].set_title("Lead Time Distribution")
axes[0, 0].set_xlabel("Lead Time (days)")

df["Division"].value_counts().plot(kind="bar", ax=axes[0, 1])
axes[0, 1].set_title("Orders by Division")
axes[0, 1].set_ylabel("Orders")

df["Ship Mode"].value_counts().plot(kind="bar", ax=axes[1, 0])
axes[1, 0].set_title("Orders by Ship Mode")
axes[1, 0].set_ylabel("Orders")

sns.boxplot(data=df, x="Division", y="Lead Time", ax=axes[1, 1])
axes[1, 1].set_title("Lead Time by Division")
axes[1, 1].set_ylabel("Lead Time (days)")

plt.tight_layout()
plt.show()

In [ ]:
print("Orders by division:")
print(df["Division"].value_counts())

print("\nOrders by region:")
print(df["Region"].value_counts())

print("\nMean lead time by ship mode:")
print(df.groupby("Ship Mode")["Lead Time"].agg(["count", "mean", "median", "std"]).round(2))

print("\nMean lead time by division:")
print(df.groupby("Division")["Lead Time"].agg(["count", "mean", "median", "std"]).round(2))

print("\nProfit margin by division:")
print(df.groupby("Division")["Profit Margin"].agg(["count", "mean", "median"]).round(3))

### Important EDA finding: severe class imbalance

The dataset is dominated by Chocolate orders:

- Chocolate: 9,844 rows
- Other: 310 rows
- Sugar: 40 rows

This means overall model performance can hide weaker performance on rare products. We should mention this limitation in the final report.

The rare products should not be deleted just because they have small sample sizes; they are part of the business dataset.

## 7. Feature preparation / encoding

Machine-learning models cannot directly consume text categories.

We use a `ColumnTransformer`:

- numerical columns → median imputation + scaling
- categorical columns → most-frequent imputation + one-hot encoding

`handle_unknown="ignore"` is important because the application may later receive a category that was not present in the training subset.

The entire preprocessing pipeline is kept together with each model so that training and future predictions use the exact same transformations.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

X = df[feature_columns]
y = df[target]

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = [col for col in X.columns if col not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Categorical features:", len(categorical_features))
print("Numerical features:", len(numeric_features))

## 8. Train the three required models

We now train the exact model families specified by the project:

### Linear Regression
A simple, interpretable baseline. It tells us how well a mostly linear relationship can explain lead time.

### Random Forest Regressor
An ensemble of decision trees. It can capture nonlinear relationships and interactions between product, geography, ship mode and transaction variables.

### Gradient Boosting Regressor
Builds trees sequentially, with each tree attempting to improve the previous model's errors. It is useful for structured tabular data.

For this stage we use a standard **random 80/20 train-test split** with `random_state=42`. A later section of the project should also discuss the unusual temporal structure of this dataset as a validation limitation.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

models = {
    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(
        n_estimators=250,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
        loss="squared_error"
    )
}

trained_models = {}
model_results = []

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    trained_models[name] = pipeline

    model_results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

results_df = pd.DataFrame(model_results).sort_values("RMSE")

print(results_df.round(3))

## Current model results

With the supplied dataset and the feature set above, the preliminary test-set results are approximately:

| Model | MAE | RMSE | R² |
|---|---:|---:|---:|
| Linear Regression | 164.55 | 179.74 | 0.543 |
| Random Forest | 117.26 | 142.46 | 0.713 |
| Gradient Boosting | 160.75 | 168.01 | 0.601 |

**Interpretation:** Random Forest currently provides the strongest test-set fit among the three models under this split.

However, this is **not yet the final model-selection step**. We should formally compare the models in Step 12 and investigate whether performance is stable across product/division groups before making the final selection.

Also, because the dataset has an unusual time structure, the report should explicitly state that the random split is useful for this scenario-modeling experiment but is not the same as a strict future-time validation.

# Checkpoint after Step 6

At this point we have completed:

- dataset loading
- dataset inspection
- cleaning and validation
- missing-value / duplicate / outlier investigation
- date processing
- lead-time target creation
- feature engineering
- EDA
- categorical/numerical preprocessing
- Linear Regression
- Random Forest
- Gradient Boosting

**Next:** Step 7 onward should focus on formal model comparison, validation diagnostics, model selection, saving the selected pipeline, and then designing the factory-simulation/recommendation layer without inventing transportation costs or factory capacity.

# Steps 7–13 — Model Validation, Selection, Factory Scenario Simulation, and Model Saving

This section continues the project after the initial EDA/model training.

**Important modeling limitation:** the supplied dataset has no historical factory-assignment field. Therefore, the ML model can predict lead time from observed order characteristics, but it cannot learn a causal factory effect. The factory engine below is therefore a **scenario-analysis layer**, not a learned factory-causal model.

The scenario engine uses the supplied product → current-factory mapping and compares factory scenarios through a transparent geographic-distance proxy. Because transportation cost and factory capacity are unavailable, it does **not** invent either quantity.


## 7. Formal model comparison

We now collect the three model results into one table and select the model using the validation metrics.

For this project, **lower MAE/RMSE is better** and **higher R² is better**.

We will use Random Forest if it remains the strongest model on the held-out test set. The selection is based on documented metrics, not on a manually chosen result.


In [ ]:
# Rebuild/collect model results from the variables created earlier in the notebook.
# This cell assumes the model-training section above created:
# lr_model, rf_model, gb_model, X_test, y_test

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

models = {
    "Linear Regression": lr_model,
    "Random Forest": rf_model,
    "Gradient Boosting": gb_model
}

comparison_rows = []

for name, model in models.items():
    pred = model.predict(X_test)
    comparison_rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred)
    })

model_comparison = pd.DataFrame(comparison_rows).sort_values(
    by=["RMSE", "MAE"], ascending=[True, True]
).reset_index(drop=True)

display(model_comparison.round(3))


In [ ]:
# Select the model with the lowest RMSE.
best_model_name = model_comparison.loc[0, "Model"]
best_model = models[best_model_name]

print("Selected model:", best_model_name)
print("\nReason: it has the lowest validation/test RMSE among the three required models.")


## 8. Error diagnostics

A good internship project should show not only the headline metrics but also whether errors behave strangely.

We inspect:
- actual vs predicted lead time
- residual distribution
- residuals against predictions


In [ ]:
import matplotlib.pyplot as plt

best_pred = best_model.predict(X_test)
residuals = y_test.to_numpy() - best_pred

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_test, best_pred, alpha=0.35)
ax.set_xlabel("Actual Lead Time")
ax.set_ylabel("Predicted Lead Time")
ax.set_title(f"Actual vs Predicted — {best_model_name}")
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(residuals, bins=30)
ax.set_xlabel("Residual (Actual - Predicted)")
ax.set_ylabel("Frequency")
ax.set_title(f"Residual Distribution — {best_model_name}")
plt.show()


## 9. Factory scenario methodology

### Why we need a separate scenario layer

There is no `Factory` field in the supplied historical data. Therefore, adding a factory label to the training data would create fabricated training information.

Instead, we estimate how a factory assignment might affect operational lead time using a **distance-based scenario proxy**.

The logic is:

1. Estimate the destination's geographic coordinates from the available `Postal Code`/city/state information where possible.
2. Calculate great-circle distance from the destination to each supplied factory.
3. Convert the distance relationship into a transparent scenario adjustment.
4. Keep the ML prediction as the baseline demand/order lead-time estimate.
5. Treat the factory-adjusted result as a **scenario estimate**, not a causal prediction.

This does **not** claim that distance equals shipping time. It is a proxy necessitated by the absence of actual factory assignment, transport time, transport cost, and capacity data.

For a production system, the next data request would be actual factory assignment and shipment origin/destination history.


In [ ]:
# Supplied factory coordinates and current product mapping.

FACTORIES = {
    "Lot's O' Nuts": (32.881893, -111.768036),
    "Wicked Choccy's": (32.076176, -81.088371),
    "Sugar Shack": (48.11914, -96.18115),
    "Secret Factory": (41.446333, -90.565487),
    "The Other Factory": (35.1175, -89.971107),
}

PRODUCT_FACTORY = {
    "Wonka Bar - Nutty Crunch Surprise": "Lot's O' Nuts",
    "Wonka Bar - Fudge Mallows": "Lot's O' Nuts",
    "Wonka Bar - Scrumdiddlyumptious": "Lot's O' Nuts",
    "Wonka Bar - Milk Chocolate": "Wicked Choccy's",
    "Wonka Bar - Triple Dazzle Caramel": "Wicked Choccy's",
    "Laffy Taffy": "Sugar Shack",
    "SweeTARTS": "Sugar Shack",
    "Nerds": "Sugar Shack",
    "Fun Dip": "Sugar Shack",
    "Everlasting Gobstopper": "Secret Factory",
    "Hair Toffee": "The Other Factory",
    "Fizzy Lifting Drinks": "Sugar Shack",
    "Lickable Wallpaper": "Secret Factory",
    "Wonka Gum": "Secret Factory",
    "Kazookles": "The Other Factory",
}

# Normalize names in the mapping using the same cleaning convention used above.
PRODUCT_FACTORY = {
    k.strip(): v for k, v in PRODUCT_FACTORY.items()
}

print("Factories:", len(FACTORIES))
print("Mapped products:", len(PRODUCT_FACTORY))


In [ ]:
# Optional geocoding support.
# The dataset does not provide latitude/longitude for customers.
# We therefore use a postal-code centroid lookup only if a geospatial
# package/data source is already available locally.
#
# To keep the project reproducible and avoid silently downloading data,
# this notebook uses a conservative scenario proxy based on the supplied
# factory coordinates and region centroids.

REGION_CENTROIDS = {
    "Central": (39.0, -98.0),
    "East": (39.0, -77.0),
    "South": (32.0, -90.0),
    "West": (37.0, -119.0),
}

def haversine_km(lat1, lon1, lat2, lon2):
    import numpy as np
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp = np.radians(lat2 - lat1)
    dl = np.radians(lon2 - lon1)
    a = np.sin(dp/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def factory_distance_km(region, factory_name):
    if region not in REGION_CENTROIDS:
        return np.nan
    lat, lon = REGION_CENTROIDS[region]
    flat, flon = FACTORIES[factory_name]
    return float(haversine_km(lat, lon, flat, flon))

distance_table = pd.DataFrame(
    [
        {
            "Region": region,
            "Factory": factory,
            "Approx_Distance_km": factory_distance_km(region, factory)
        }
        for region in REGION_CENTROIDS
        for factory in FACTORIES
    ]
)

display(distance_table.head(10))


### Scenario adjustment rule

Because there is no observed factory-level lead-time data, we avoid claiming a statistically learned factory coefficient.

Instead, we use a conservative **relative distance adjustment**:

`scenario lead time = baseline lead time × (1 + alpha × relative distance difference)`

where `alpha` is deliberately small.

This is only a transparent what-if proxy. The final dashboard should label these values **Scenario Estimate**, not "predicted factory lead time."

The recommendation engine will still use the ML model for the baseline and historical profitability, while the factory layer evaluates the operational scenario.


In [ ]:
SCENARIO_ALPHA = 0.15  # conservative scenario sensitivity; NOT a learned coefficient

def scenario_factory_table(product, region, baseline_lead_time):
    if product not in PRODUCT_FACTORY:
        raise ValueError(f"No current factory mapping found for product: {product}")
    if region not in REGION_CENTROIDS:
        raise ValueError(f"No region centroid available for region: {region}")

    current_factory = PRODUCT_FACTORY[product]
    distances = {
        f: factory_distance_km(region, f)
        for f in FACTORIES
    }

    current_distance = distances[current_factory]
    rows = []

    for factory, distance in distances.items():
        relative_change = (distance - current_distance) / max(current_distance, 1e-9)
        estimated_lead = baseline_lead_time * (1 + SCENARIO_ALPHA * relative_change)

        improvement = baseline_lead_time - estimated_lead
        improvement_pct = (improvement / baseline_lead_time) * 100

        rows.append({
            "Factory": factory,
            "Current Factory": factory == current_factory,
            "Approx Distance (km)": distance,
            "Scenario Lead Time (days)": estimated_lead,
            "Lead-Time Change (days)": estimated_lead - baseline_lead_time,
            "Lead-Time Improvement (%)": improvement_pct
        })

    return pd.DataFrame(rows).sort_values(
        "Scenario Lead Time (days)"
    ).reset_index(drop=True)

# Example using the first available product and region.
example_product = list(PRODUCT_FACTORY.keys())[0]
example_region = list(REGION_CENTROIDS.keys())[0]

example_baseline = float(np.median(y_test))
scenario_factory_table(example_product, example_region, example_baseline)


## 10. Profitability and recommendation logic

The dataset provides `Sales`, `Gross Profit`, and `Cost`, so we can calculate historical gross margin.

We do **not** calculate transportation savings because no transportation cost is supplied.

The recommendation score combines:
- operational improvement
- historical gross margin
- a risk penalty

The speed/profit weighting is exposed as a parameter for the future Streamlit application.


In [ ]:
# Build historical product-level profitability statistics.

df_work = df.copy()

df_work["Profit_Margin"] = np.where(
    df_work["Sales"] != 0,
    df_work["Gross Profit"] / df_work["Sales"],
    np.nan
)

product_profit = (
    df_work.groupby("Product Name", as_index=False)
    .agg(
        Avg_Sales=("Sales", "mean"),
        Avg_Gross_Profit=("Gross Profit", "mean"),
        Avg_Profit_Margin=("Profit_Margin", "mean"),
        Orders=("Order ID", "nunique")
    )
)

display(product_profit.sort_values("Avg_Gross_Profit", ascending=False).head(10))


In [ ]:
def recommendation_for(product, region, ship_mode=None, speed_weight=0.7):
    """
    Generate an explainable factory scenario recommendation.

    speed_weight:
        1.0 = operational speed dominates
        0.0 = profitability dominates

    Important:
        Factory lead times are scenario estimates because historical
        factory assignments are not present in the dataset.
    """

    if product not in df_work["Product Name"].unique():
        raise ValueError(f"Unknown product: {product}")

    subset = df_work[df_work["Product Name"] == product].copy()

    if region is not None:
        region_subset = subset[subset["Region"] == region]
        if len(region_subset) > 0:
            subset = region_subset

    if ship_mode is not None:
        mode_subset = subset[subset["Ship Mode"] == ship_mode]
        if len(mode_subset) > 0:
            subset = mode_subset

    if len(subset) == 0:
        raise ValueError("No historical observations match the selected filters.")

    # Baseline: median observed lead time for the selected scenario.
    baseline = float(subset["Lead Time"].median())

    current_factory = PRODUCT_FACTORY.get(product)
    if current_factory is None:
        raise ValueError("Product does not have a supplied factory mapping.")

    scenarios = scenario_factory_table(product, region, baseline)

    # Historical profitability.
    margin = float(subset["Profit_Margin"].mean())
    orders = int(subset["Order ID"].nunique())

    # Normalize improvement to 0–1 across candidate factories.
    imp = scenarios["Lead-Time Improvement (%)"].clip(lower=0)
    max_imp = max(float(imp.max()), 1e-9)
    scenarios["Speed_Score"] = imp / max_imp

    # Profit score: use historical margin as the economic indicator.
    # It is the same across factories because factory-specific profitability
    # is unavailable.
    scenarios["Profit_Score"] = margin

    # Risk: fewer observations means less historical support.
    if orders < 10:
        risk = "High"
    elif orders < 30:
        risk = "Medium"
    else:
        risk = "Lower"

    scenarios["Risk"] = risk
    scenarios["Decision_Score"] = (
        speed_weight * scenarios["Speed_Score"]
        + (1 - speed_weight) * scenarios["Profit_Score"]
    )

    ranked = scenarios.sort_values(
        ["Decision_Score", "Lead-Time Improvement (%)"],
        ascending=[False, False]
    ).reset_index(drop=True)

    # Avoid recommending a switch when there is no improvement.
    current_row = scenarios[scenarios["Factory"] == current_factory].iloc[0]
    best_row = ranked.iloc[0]

    if best_row["Factory"] == current_factory or best_row["Lead-Time Improvement (%)"] <= 0:
        recommendation = "Keep current factory"
    else:
        recommendation = f"Consider scenario: {best_row['Factory']}"

    return {
        "product": product,
        "region": region,
        "ship_mode": ship_mode,
        "current_factory": current_factory,
        "baseline_lead_time_days": baseline,
        "profit_margin": margin,
        "historical_orders": orders,
        "risk": risk,
        "recommendation": recommendation,
        "scenarios": ranked
    }

# Example
demo = recommendation_for(
    product=example_product,
    region=example_region,
    ship_mode=None,
    speed_weight=0.7
)

print("Recommendation:", demo["recommendation"])
print("Current factory:", demo["current_factory"])
print("Baseline lead time:", round(demo["baseline_lead_time_days"], 2), "days")
print("Historical profit margin:", round(demo["profit_margin"] * 100, 2), "%")
print("Risk:", demo["risk"])
display(demo["scenarios"])


## 11. Model interpretability

For the tree-based models, feature importance gives a useful explanation of what the model relied on.

This is **not causal importance**. It tells us which input variables contributed most to prediction within the trained model.


In [ ]:
# Show feature importance if the selected model exposes it.
if hasattr(best_model, "feature_importances_"):
    try:
        feature_names = preprocessor.get_feature_names_out()
        importance_df = pd.DataFrame({
            "Feature": feature_names,
            "Importance": best_model.feature_importances_
        }).sort_values("Importance", ascending=False)

        display(importance_df.head(20))
    except Exception as e:
        print("Could not recover transformed feature names:", e)
elif hasattr(best_model, "coef_"):
    try:
        feature_names = preprocessor.get_feature_names_out()
        coef_df = pd.DataFrame({
            "Feature": feature_names,
            "Coefficient": best_model.coef_
        })
        coef_df["Absolute_Coefficient"] = coef_df["Coefficient"].abs()
        display(coef_df.sort_values("Absolute_Coefficient", ascending=False).head(20))
    except Exception as e:
        print("Could not recover transformed feature names:", e)


## 12. Save the trained model and preprocessing pipeline

The Streamlit application should not retrain the model every time the user opens the dashboard.

Therefore we save:
- the preprocessing transformer
- the selected trained model
- the factory mappings and scenario metadata

If the notebook trained the model as a single sklearn Pipeline, saving that pipeline is preferable because preprocessing and prediction remain together.


In [ ]:
from pathlib import Path
import joblib

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Try to save a single pipeline if the notebook has one.
# If the model was trained separately from preprocessing, save both.
if "model_pipeline" in globals():
    artifact = {
        "pipeline": model_pipeline,
        "model_name": best_model_name,
        "scenario_alpha": SCENARIO_ALPHA,
        "factories": FACTORIES,
        "product_factory": PRODUCT_FACTORY
    }
else:
    artifact = {
        "preprocessor": preprocessor,
        "model": best_model,
        "model_name": best_model_name,
        "scenario_alpha": SCENARIO_ALPHA,
        "factories": FACTORIES,
        "product_factory": PRODUCT_FACTORY
    }

model_path = MODEL_DIR / "lead_time_model.joblib"
joblib.dump(artifact, model_path)

print("Saved:", model_path.resolve())


# What we have completed

At this point the notebook covers the core data-science portion:

**Dataset → Cleaning → Feature Engineering → EDA → Three ML Models → Evaluation → Model Selection → Scenario Engine → Recommendation Logic → Saved Model**

The remaining implementation is the application/report layer:

1. Build `app/app.py`
2. Load the saved model
3. Add Product / Region / Ship Mode controls
4. Add Speed ↔ Profit priority
5. Display factory scenario table
6. Add charts
7. Add risk/explanation cards
8. Write README
9. Produce the research paper
10. Produce the executive summary

### Important limitation to state in the report

The dataset contains no:
- historical factory assignment
- factory capacity
- transportation cost
- transportation distance from customer to factory
- factory-specific production time

Therefore the project can demonstrate **lead-time prediction and explainable factory what-if analysis**, but it cannot claim that the ML model has empirically learned the causal effect of moving production between factories.

That distinction makes the project more credible, not less.


# Final Implementation — Streamlit

The project folder now contains a standalone training script and Streamlit dashboard.

From the project root:

```bash
pip install -r requirements.txt
python models/train_model.py
streamlit run app/app.py
```

The dashboard is designed to be presentation-friendly while keeping the factory limitation explicit.
